In [1]:
import os
import pandas as pd
import numpy as np
import igraph
import gradco as gradco
import networkx as nx

In [2]:
def process_graphlet(k, counter, dir_path):
    #get orbit adjacency matrix for graphlet k
    temp = counter.get_graphlet_adjacency(k)  
    M = temp.todense()                        
    graphlet_df = pd.DataFrame(M)
    graphlet_df.to_csv(f'{dir_path}/G{k}.csv', index=False, header=False)  


In [3]:
dir_path=os.path.join(os.getcwd(), 'graphlets')

### Karate Club Network

In [4]:
G = nx.karate_club_graph()
club_mapping = {'Mr. Hi': 0, 'Officer': 1}  #assign numeric values
real_membership = [club_mapping[G.nodes[node]['club']] for node in G.nodes()]

In [5]:
#get adjacency matrix and binarize it
ADJ = nx.adjacency_matrix(G).todense()
ADJ[ADJ != 0] = 1 

#count graphlet orbits and save adjacency matrices for G0-G8
counter = gradco.Counter(ADJ)
counter.count()
for k in range(9):
    process_graphlet(k, counter, dir_path)


Computing A4_4
BUILDING DIGRAPH
BRUTE FORCE
Execution Time: 0 minutes and 0 seconds
COMPUTING REDUNDANCY MATRICES
Execution Time: 0 minutes and 0 seconds
COMPUTING ADJACENCY MATRICES
Execution Time: 0 minutes and 0 seconds
CONVERTING TO NUMPY ARRAYS
Execution Time: 0 minutes and 0 seconds
TOTAL TIME
Execution Time: 0 minutes and 0 seconds
Computation time = 0m 0s


In [6]:
def check_communities(current_graph, real_membership):
    comm_detect_output=igraph.Graph.community_fastgreedy(current_graph, weights='weight').as_clustering()
    detected_membership=comm_detect_output.membership
    modularity=comm_detect_output.modularity
    nmi_metric=igraph.compare_communities(detected_membership, real_membership, method='nmi')
    ari_metric=igraph.compare_communities(detected_membership, real_membership, method='adjusted_rand_index')
    return nmi_metric, ari_metric, modularity


In [7]:
rows = []

In [8]:
for graphlet in range(9):
    if graphlet == 0: #baseline G0
        A0 = pd.read_csv(f'{dir_path}/G{graphlet}.csv', header=None)
        current_graph = igraph.Graph.Weighted_Adjacency(A0, mode="undirected", attr="weight")
    else: #G1-G8
        currentMAT = pd.read_csv(f'{dir_path}/G{graphlet}.csv', header=None)
        #normalize
        currentMAT=currentMAT/np.max(currentMAT.values)  
        #add baseline matrix
        currentMAT=currentMAT+A0 
        #construct graph
        current_graph=igraph.Graph.Weighted_Adjacency(currentMAT, mode="undirected", attr="weight")

    # apply FastGreedy community detection and compute metrics (NMI, ARI, Modularity)
    nmi_metric, ari_metric, modularity = check_communities(current_graph, real_membership)
    rows.append({'Graphlet': f'G{graphlet}', 'Modularity': modularity, 'ARI': ari_metric, 'NMI': nmi_metric})

final_tbl = pd.DataFrame(rows, columns=['Graphlet', 'Modularity', 'ARI', 'NMI'])


In [9]:
final_tbl

,Graphlet,Modularity,ARI,NMI
0,G0,0.380671,0.568439,0.564607
1,G1,0.331404,0.508864,0.600011
2,G2,0.400933,0.406449,0.500483
3,G3,0.258284,0.443007,0.486719
4,G4,0.327255,0.709235,0.705338
5,G5,0.293377,0.571984,0.494943
6,G6,0.356293,0.709235,0.705338
7,G7,0.421180,0.407099,0.492623
8,G8,0.431880,0.483076,0.592527


In [10]:
final_tbl.to_csv('final_results.csv', index=False)